# **Modelo2_Daniele**

Dataset: **dataset_caracteristicas_train_V1_ALL.csv**

## ***¿Qué hacemos en este notebook?***

En este notebook **optimizamos los hiperparámetros del XGBoost** sobre los **4.000 datos de entrenamiento disponibles**.

### ***¿Por qué XGBoost?***

La diferencia entre XGBoost (0.9452) y Red Neuronal (0.9466) es de **0.0014** (ruiso estadistico). Elegimos el XGBoost porque:

- Con solo 4.000 muestras, el XGBoost está en su régimen óptimo (diseñado para datasets tabulares de tamaño pequeño-mediano). La Red Neuronal necesita decenas de miles de muestras para demostrar su ventaja real.
- Su espacio de hiperparámetros responde de forma más predecible al tuning con datos limitados.

### ***¿Por qué 4.000 muestras y no 2.240 (el "plateau" del Modelo1)***?

Las learning curves del Modelo1 detectaron una **desaceleración** en N = 2.240 usando el criterio ΔVal < 0.005, pero **no un plateau ideal definitivo** que nos garantizara que el F1-score con 4.000 datos fuera igual al F1-score con 2.240 muestras: todos los modelos siguen mejorando significativamente hasta las 4.000 muestras disponibles (el XGBoost gana +0.0124 en F1 de N = 2.240 a N = 4.000).

Esta limitación proviene del **desbalanceo original del dataset** (22.800 spoof vs 2.580 bonafide): habiendo realizado un undersampling a 5.000 muestras garantizando el 50% de muestras spoof y el 50% de muestras bobafide, el conjunto de entrenamiento está compuesto de 4.000 muestras. No existe un N optimo < 4.000 que dé el mismo F1 que N = 4.000. 

**CONCLUSIÓN:** Optimizamos sobre los **4.000 datos**.

### **OBJETIVO**

Superar el F1-score del XGBoost implementado en el Modelo1: **0.9452** (evaluado con CV 5-fold sobre 4.000 muestras).

El test sellado (1.000 muestras) **no se toca**, se reserva para el `Modelo3_Daniele.ipynb`.

### ***Bloque 1: configuración global***

In [23]:
# Bloque 1: configuración global

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import (
    train_test_split, StratifiedKFold,
    cross_validate, RandomizedSearchCV
)
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score, roc_auc_score
from scipy.stats import randint, uniform

try:
    from xgboost import XGBClassifier
    print('XGBoost disponible')
except ImportError:
    raise ImportError('XGBoost no instalado. Ejecuta: pip install xgboost')

sns.set_theme(style='whitegrid', palette='Set2', font_scale=1.15)
plt.rcParams['figure.dpi']       = 110
plt.rcParams['axes.titleweight'] = 'bold'

# --- Parámetros globales (IDÉNTICOS al EDA y Modelo1) ---
RANDOM_STATE = 42
N_PER_CLASS  = 2500
TEST_SIZE    = 0.20
N_FOLDS      = 5
DATA_PATH    = '../Obtencion_Metricas/dataset_caracteristicas_train_V1_ALL.csv'

print(f'RANDOM_STATE={RANDOM_STATE} | N_PER_CLASS={N_PER_CLASS} | N_FOLDS={N_FOLDS}')
print(f'Entrenamiento sobre los 4.000 datos de train')

XGBoost disponible
RANDOM_STATE=42 | N_PER_CLASS=2500 | N_FOLDS=5
Entrenamiento sobre los 4.000 datos de train


## **SECCIÓN 1**: reproducimos el mismo pipeline

El pipeline es idéntico al Modelo1: misma semilla, mismo balanceo, mismo split 80/20 estratificado.  
El test (1.000 muestras) no se toca en este notebook, lo usaremos para testear el modelo optimizado en el paso sucesivo. 
Trabajamos con los 4.000 datos de train completos.

### ***Bloque 2: pipeline***

In [24]:
# Bloque 2: pipeline

df_full     = pd.read_csv(DATA_PATH)
df_bonafide = df_full[df_full['label'] == 'bonafide'].copy()
df_spoof    = df_full[df_full['label'] == 'spoof'].copy()

n_bonafide    = min(len(df_bonafide), N_PER_CLASS)
df_bon_sample = df_bonafide.sample(n=n_bonafide, random_state=RANDOM_STATE)

attack_counts = df_spoof['attack_id'].value_counts()
proportions   = attack_counts / attack_counts.sum()
n_per_attack  = (proportions * N_PER_CLASS).astype(int)
deficit       = N_PER_CLASS - n_per_attack.sum()
for atk in n_per_attack.nlargest(abs(deficit)).index:
    n_per_attack[atk] += int(np.sign(deficit))

spoof_samples = []
for attack_id, n in n_per_attack.items():
    subset = df_spoof[df_spoof['attack_id'] == attack_id]
    spoof_samples.append(subset.sample(n=min(n, len(subset)), random_state=RANDOM_STATE))

df_balanced = pd.concat([df_bon_sample, pd.concat(spoof_samples)]).sample(
    frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

FEATURE_COLS = [c for c in df_balanced.columns if c not in ['file_name', 'attack_id', 'label']]
X            = df_balanced[FEATURE_COLS]
y            = df_balanced['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)

le          = LabelEncoder().fit(['bonafide', 'spoof'])
y_train_enc = le.transform(y_train)
y_test_enc  = le.transform(y_test)

print(f'Train: {X_train.shape[0]:,} filas | Test: {X_test.shape[0]:,} filas')
print(f'Número de features: {len(FEATURE_COLS)}')
print()
print('Usamos los 4.000 datos de train COMPLETOS para la búsqueda de hiperparámetros')

Train: 4,000 filas | Test: 1,000 filas
Número de features: 34

Usamos los 4.000 datos de train COMPLETOS para la búsqueda de hiperparámetros


## **SECCIÓN 2**: búsqueda de Hiperparámetros XGBoost (RandomizedSearchCV)

Aplicamos `RandomizedSearchCV` con **n_iter=50** combinaciones y CV 5-fold sobre los **4.000 datos de train**.  

**Hiperparámetros** más influyentes del XGBoost:

| Hiperparámetro | Rango | Efecto |
|---|---|---|
| **n_estimators** | [100, 700] | Número de árboles |
| **max_depth** | [3, 9] | Profundidad máxima de cada árbol |
| **learning_rate** | continuo [0.01, 0.30] | Tasa de aprendizaje |
| **subsample** | continuo [0.60, 1.00] | Fracción de muestras por árbol |
| **colsample_bytree** | continuo [0.50, 1.00] | Fracción de features por árbol |
| **reg_alpha** | continuo [0, 1] | Regularización L1 |
| **reg_lambda** | continuo [0.5, 5] | Regularización L2 |
| **min_child_weight** | [1, 10] | Mínimo de muestras en hoja |

Con ***n_iter=50*** y CV 5-fold realizamos **250 entrenamientos** en total.  
El modelo base que implementamos en el código anterior usaba: **n_estimators=200, max_depth=5, learning_rate=0.1, subsample=0.8, colsample_bytree=0.8**.  

### ***BLOQUE 3: Randomizesearch CV XGBoost***

In [25]:
#  BLOQUE 3: RANDOMIZEDSEARCH CV sobre XGBoost

xgb_base = XGBClassifier(
    objective='binary:logistic',
    eval_metric='logloss',
    use_label_encoder=False,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

param_dist_xgb = {
    'n_estimators':      randint(100, 700),
    'max_depth':         randint(3, 10),
    'learning_rate':     uniform(0.01, 0.29),
    'subsample':         uniform(0.60, 0.40),
    'colsample_bytree':  uniform(0.50, 0.50),
    'reg_alpha':         uniform(0.0, 1.0),
    'reg_lambda':        uniform(0.5, 4.5),
    'min_child_weight':  randint(1, 11),
}

cv_search = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)

search_xgb = RandomizedSearchCV(
    xgb_base,
    param_distributions=param_dist_xgb,
    n_iter=50,
    scoring='f1',
    cv=cv_search,
    refit=True,
    n_jobs=-1,
    random_state=RANDOM_STATE,
    verbose=0
)

t0 = time.time()
search_xgb.fit(X_train, y_train_enc)
elapsed = time.time() - t0

print(f'XGBoost - F1 modelo base       (CV, N = 4.000): 0.9452')
print(f'XGBoost - F1 modelo optimizado (CV, N = 4.000): {search_xgb.best_score_:.4f}')
print(f'Diferencia F1-score: {search_xgb.best_score_ - 0.9452:+.4f}')
print()
print('Valores hiperparámetros óptimos encontrados:')
for k, v in sorted(search_xgb.best_params_.items()):
    print(f'  {k:<22}: {v}')

best_xgb = search_xgb.best_estimator_

XGBoost - F1 modelo base       (CV, N = 4.000): 0.9452
XGBoost - F1 modelo optimizado (CV, N = 4.000): 0.9484
Diferencia F1-score: +0.0032

Valores hiperparámetros óptimos encontrados:
  colsample_bytree      : 0.6379995910112717
  learning_rate         : 0.09591931665418388
  max_depth             : 7
  min_child_weight      : 1
  n_estimators          : 620
  reg_alpha             : 0.7722447692966574
  reg_lambda            : 1.3942205669037757
  subsample             : 0.602208846849441


## **SECCIÓN 3**: Validación Cruzada Completa del XGBoost Optimizado

Calculamos todas las métricas y el GAP train-validación para detectar overfitting.

### ***Bloque 4: validación cruzada del mejor XGBoost***

In [26]:
#  BLOQUE 4: metricas del mejor XGBoost

cv_final = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)

scoring = {
    'accuracy':  'accuracy',
    'precision': 'precision',
    'recall':    'recall',
    'f1':        'f1',
    'roc_auc':   'roc_auc'
}

cv_opt = cross_validate(
    best_xgb, X_train, y_train_enc,
    cv=cv_final,
    scoring=scoring,
    return_train_score=True,
    n_jobs=-1
)

m2_val_f1   = cv_opt['test_f1'].mean()
m2_train_f1 = cv_opt['train_f1'].mean()
m2_gap      = m2_train_f1 - m2_val_f1

print(f'Resultados CV ({N_FOLDS}-fold) - XGBoost optimizado (N = 4.000)')
print(f'{"Métrica":<20} {"Train":>10} {"Val":>10} {"GAP":>10}')
print('-' * 52)
for metric in ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']:
    tr = cv_opt[f'train_{metric}'].mean()
    va = cv_opt[f'test_{metric}'].mean()
    print(f'{metric:<20} {tr:>10.4f} {va:>10.4f} {tr-va:>+10.4f}')
print(f'\nGAP F1 (train-val): {m2_gap:+.4f}')

Resultados CV (5-fold) - XGBoost optimizado (N = 4.000)
Métrica                   Train        Val        GAP
----------------------------------------------------
accuracy                 1.0000     0.9490    +0.0510
precision                1.0000     0.9601    +0.0399
recall                   1.0000     0.9370    +0.0630
f1                       1.0000     0.9484    +0.0516
roc_auc                  1.0000     0.9869    +0.0131

GAP F1 (train-val): +0.0516


## **SECCIÓN 4**: Comparación XGBoost base vs XGBoost optimizado

Comparamos el XGBoost base del con el XGBoost optimizado, **ambos evaluados con CV 5-fold sobre los mismos 4.000 datos de train**.

Esta comparación es **justa** porque:
- El mismo conjunto de datos (4.000 muestras de train);
- El mismo protocolo de evaluación (CV estratificado 5-fold);
- La única variable que cambia son los **hiperparámetros**.

Cualquier mejora en F1 se debe exclusivamente al tuning de hiperparámetros.

In [27]:
#  BLOQUE 5: Comparación XGBoost base vs XGBoost optimizado

cv_kfold    = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
scoring_cmp = {'f1': 'f1', 'roc_auc': 'roc_auc'}

# Baseline: XGBoost con los mismos hiperparámetros que en Modelo1
xgb_baseline = XGBClassifier(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    objective='binary:logistic',
    eval_metric='logloss',
    use_label_encoder=False,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

cv_base = cross_validate(
    xgb_baseline, X_train, y_train_enc,
    cv=cv_kfold, scoring=scoring_cmp, return_train_score=True, n_jobs=-1
)

m1_val_f1   = cv_base['test_f1'].mean()
m1_train_f1 = cv_base['train_f1'].mean()
m1_gap      = m1_train_f1 - m1_val_f1
m1_val_auc  = cv_base['test_roc_auc'].mean()
m2_val_auc  = cv_opt['test_roc_auc'].mean()

delta_f1  = m2_val_f1 - m1_val_f1
delta_auc = m2_val_auc - m1_val_auc
delta_gap = m2_gap     - m1_gap

print(f'Comparación XGBoost Base vs Optimizado (CV 5-fold, N = 4.000 muestras)')
print(f'{"":32} {"Val F1":>10} {"Val AUC":>10} {"GAP F1":>10}')
print('-' * 72)
print(f'{"XGBoost base (modelo 1)":<32} {m1_val_f1:>10.4f} {m1_val_auc:>10.4f} {m1_gap:>+10.4f}')
print(f'{"XGBoost optimizado (modelo2)":<32} {m2_val_f1:>10.4f} {m2_val_auc:>10.4f} {m2_gap:>+10.4f}')
print(f'{"Diferencia":<32} {delta_f1:>+10.4f} {delta_auc:>+10.4f} {delta_gap:>+10.4f}')

Comparación XGBoost Base vs Optimizado (CV 5-fold, N = 4.000 muestras)
                                     Val F1    Val AUC     GAP F1
------------------------------------------------------------------------
XGBoost base (modelo 1)              0.9452     0.9866    +0.0548
XGBoost optimizado (modelo2)         0.9484     0.9869    +0.0516
Diferencia                          +0.0032    +0.0003    -0.0032


## **CONCLUSIONES**

- **Elegimos el XGBoost** como modelo a optimizar: la diferencia de F1 respecto a la Red Neuronal (0.0014) está dentro del ruido estadístico del CV y el XGBoost es estructuralmente mejor adaptado a 4.000 muestras de entrenamiento;

- **Usamos los 4.000 datos de train completos**: no existe un plateau ideal dentro del rango disponible. El criterio ΔVal < 0.005 del Modelo1 detectó una desaceleración puntual en N = 2.240, no una convergencia. Esta limitación proviene del desbalanceo original del dataset (22.800 spoof vs 2.580 bonafide);

- **Aplicamos `RandomizedSearchCV`** (50 iteraciones, CV 5-fold) al XGBoost sobre los 4.000 datos.
El espacio de hiperparámetros del XGBoost incluye parámetros continuos (learning_rate, subsample, colsample_bytree, reg_alpha, reg_lambda) y discretos (n_estimators, max_depth). Un grid exhaustivo implicaría miles de combinaciones. `RandomizedSearchCV` con `n_iter=50` muestrea aleatoriamente el espacio y encuentra soluciones próximas al óptimo en tiempo razonable.

- **Comparación justa**: XGBoost baseline vs XGBoost optimizado, **ambos** evaluados con CV 5-fold sobre los mismos 4.000 datos. La única variable que cambia son los hiperparámetros.

- El conjunto de test se quedó sellado (1.000 muestras) y **no se tocó**.

** ***El modelo optimizado es mejor del modelo base?*** 

La mejora del F1-score en validación es **pequeña** (+0.0032), pero va en la dirección esperada tras el tuning.

El **AUC** también sube un poco (+0.0003).

El **GAP train–val** del optimizado es menor, así que la brecha entre entrenamiento y validación no empeora.